In [ ]:
import re
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import tensorflow as tf
import pickle
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout,Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

In [ ]:
import nltk
import spacy
nlp = spacy.load("en_core_web_sm")
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
DATA_PATH = '/content/drive/MyDrive/seminar2/dataset'
EMBEDDING_DIM = 256
MAX_LEN = 128
HIDDEN_DIM = 256
BATCH_SIZE = 256
EPOCHS = 20
DROPOUT_RATE=0.2
RANDOM_SEED = 42
LABELS = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']

In [ ]:
# load data
df=pd.read_csv('/content/drive/MyDrive/seminar2/dataset/train.csv')

In [ ]:
stop_words = nltk.corpus.stopwords.words("english")
def clean_text(text):
    text = text.lower()
    text=re.sub(r"http\S+" , " " , text)
    text = re.sub('\n',' ',text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text) # bỏ ký tự đặc biệt
    text = re.sub(r"\s+", " ", text).strip() # bỏ khoảng trắng thừa
    text = ' '.join([word for word in text.split() if not word in stop_words])
    return text

In [ ]:
df['clean_text']=df['comment_text'].map(clean_text)

In [ ]:
df[['clean_text','comment_text']]

,clean_text,comment_text
0,maybe ward driven turn,Or maybe after Ward's been driven off it's my ...
1,omitted foreshadowed discussion cold war next ...,which I omitted foreshadowed the discussion ab...
2,bottom 1 paragraph entire poem song believe co...,At the bottom there was only 1 paragraph from ...
3,love anime animes suck idiot 68 84 225 186,You LOVE Anime!? There are some ANIMES out the...
4,seriously realise difference free content non ...,Do you seriously not realise that there's a di...
...,...,...
143608,say stuff like transcoding templates really ta...,"""\n Now when you say stuff like """"'transcodin..."
143609,agree aleta merit discussion blp consideration...,"""\nI agree with Aleta that it does merit discu..."
143610,blocked blocked consider immature aggression a...,Blocked\nI am blocked by what I consider to be...
143611,joey publicist asked jason make change member ...,I am Joey's publicist. I asked Jason to make t...


In [ ]:
df['clean_text'].map(lambda x: len(x.split())).mean()

np.float64(35.38415742307451)

In [ ]:
tokenizer = Tokenizer()

In [ ]:
tokenizer.fit_on_texts(df['clean_text'])

In [ ]:
with open('/content/drive/MyDrive/seminar2tokenizer.pickle', 'rb') as f:
    tokenizer=pickle.load( f)

In [ ]:
train_df,valid_df=train_test_split(df,test_size=1/9,random_state=RANDOM_SEED)

In [ ]:
class_weight=1/train_df[LABELS].sum().values

In [ ]:
class_weight/=sum(class_weight)

In [ ]:
class_weight={i:v.item() for i,v in enumerate(class_weight)}

In [ ]:
x_train_seq = tokenizer.texts_to_sequences(train_df['clean_text'])
x_valid_seq = tokenizer.texts_to_sequences(valid_df['clean_text'])
x_train_pad = pad_sequences(x_train_seq,maxlen=MAX_LEN, padding='post',truncating='post')
x_valid_pad = pad_sequences(x_valid_seq,maxlen=MAX_LEN, padding='post',truncating='post')

In [ ]:
vocab_size = len(tokenizer.word_index)+1

In [ ]:
with open('/content/drive/MyDrive/seminar2tokenizer.pickle', 'wb') as f:
    pickle.dump(tokenizer, f)

In [ ]:
y_train = train_df[LABELS].values
y_val = valid_df[LABELS].values

In [ ]:
vocab_size

162820

In [ ]:
for l in LABELS:
    print(df[l].value_counts())

toxic
0    129820
1     13793
Name: count, dtype: int64
severe_toxic
0    142167
1      1446
Name: count, dtype: int64
obscene
0    135973
1      7640
Name: count, dtype: int64
threat
0    143182
1       431
Name: count, dtype: int64
insult
0    136505
1      7108
Name: count, dtype: int64
identity_hate
0    142326
1      1287
Name: count, dtype: int64


In [ ]:
def build_model(embedding_matrix=None, vocab_size=100000, embedding_dim=EMBEDDING_DIM, maxlen=MAX_LEN,nb_layer=1):
    model = Sequential()
    model.add(Input(shape=(maxlen,)))
    if embedding_matrix is not None:
        nb_words = embedding_matrix.shape[0]
        model.add(Embedding(nb_words, embedding_dim, weights=[embedding_matrix], trainable=False))
    else:
        model.add(Embedding(vocab_size, embedding_dim, trainable=True))
    # model.add(Dropout(0.2))
    for _ in range(nb_layer-1):
        model.add(Bidirectional(LSTM(HIDDEN_DIM, return_sequences=True, dropout=DROPOUT_RATE, recurrent_dropout=DROPOUT_RATE)))
    model.add(Bidirectional(LSTM(HIDDEN_DIM, return_sequences=False, dropout=DROPOUT_RATE, recurrent_dropout=DROPOUT_RATE)))
    model.add(Dense(1024, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(len(LABELS), activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy', tf.keras.metrics.F1Score(average='macro',threshold=0.5)])
    model.summary()
    return model

In [ ]:
model=build_model(vocab_size=vocab_size,nb_layer=4)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 128, 256)       │    41,681,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 128, 512)       │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 128, 512)       │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_6 (Bidirectional) │ (None, 128, 512)       │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_7 (Bidirectional) │ (None, 512)            │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1024)           │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │         6,150 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,988,742 (183.06 MB)

 Trainable params: 47,988,742 (183.06 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
checkpoint = ModelCheckpoint('/content/drive/MyDrive/seminar2/best_4_weight_bilstm_toxic.h5', monitor='val_f1_score', mode='max', save_best_only=True, verbose=1)
# early = EarlyStopping(monitor='val_f1_score', mode='max', patience=3, verbose=1, restore_best_weights=True)
# reduce_lr = ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=2, verbose=1)

In [ ]:
history = model.fit(
x_train_pad, y_train,
validation_data=(x_valid_pad , y_val),
class_weight=class_weight,
batch_size=BATCH_SIZE,
epochs=EPOCHS,
callbacks=[checkpoint]
)

Epoch 1/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8147 - f1_score: 0.1432 - loss: 0.0027
Epoch 1: val_f1_score improved from -inf to 0.35736, saving model to /content/drive/MyDrive/seminar2/best_4_weight_bilstm_toxic.h5


499/499 ━━━━━━━━━━━━━━━━━━━━ 1615s 3s/step - accuracy: 0.8148 - f1_score: 0.1435 - loss: 0.0027 - val_accuracy: 0.4402 - val_f1_score: 0.3574 - val_loss: 0.0592
Epoch 2/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7856 - f1_score: 0.3910 - loss: 0.0012
Epoch 2: val_f1_score improved from 0.35736 to 0.38897, saving model to /content/drive/MyDrive/seminar2/best_4_weight_bilstm_toxic.h5


499/499 ━━━━━━━━━━━━━━━━━━━━ 1594s 3s/step - accuracy: 0.7857 - f1_score: 0.3910 - loss: 0.0012 - val_accuracy: 0.3023 - val_f1_score: 0.3890 - val_loss: 0.0576
Epoch 3/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8243 - f1_score: 0.4150 - loss: 0.0011
Epoch 3: val_f1_score improved from 0.38897 to 0.45301, saving model to /content/drive/MyDrive/seminar2/best_4_weight_bilstm_toxic.h5


499/499 ━━━━━━━━━━━━━━━━━━━━ 1586s 3s/step - accuracy: 0.8244 - f1_score: 0.4150 - loss: 0.0011 - val_accuracy: 0.9944 - val_f1_score: 0.4530 - val_loss: 0.0553
Epoch 4/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8981 - f1_score: 0.4270 - loss: 9.4680e-04
Epoch 4: val_f1_score improved from 0.45301 to 0.46079, saving model to /content/drive/MyDrive/seminar2/best_4_weight_bilstm_toxic.h5


499/499 ━━━━━━━━━━━━━━━━━━━━ 1575s 3s/step - accuracy: 0.8981 - f1_score: 0.4270 - loss: 9.4678e-04 - val_accuracy: 0.9930 - val_f1_score: 0.4608 - val_loss: 0.0525
Epoch 5/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9200 - f1_score: 0.4480 - loss: 8.7617e-04
Epoch 5: val_f1_score did not improve from 0.46079
499/499 ━━━━━━━━━━━━━━━━━━━━ 1575s 3s/step - accuracy: 0.9200 - f1_score: 0.4480 - loss: 8.7620e-04 - val_accuracy: 0.9931 - val_f1_score: 0.4193 - val_loss: 0.0561
Epoch 6/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8900 - f1_score: 0.4651 - loss: 8.3771e-04
Epoch 6: val_f1_score did not improve from 0.46079
499/499 ━━━━━━━━━━━━━━━━━━━━ 1560s 3s/step - accuracy: 0.8899 - f1_score: 0.4651 - loss: 8.3772e-04 - val_accuracy: 0.9941 - val_f1_score: 0.4526 - val_loss: 0.0555
Epoch 7/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8703 - f1_score: 0.4645 - loss: 8.1164e-04
Epoch 7: val_f1_score did not improve from 0.46079
499/499 ━━━━━━━━━━━━━━━━━━━━

499/499 ━━━━━━━━━━━━━━━━━━━━ 1557s 3s/step - accuracy: 0.9800 - f1_score: 0.5686 - loss: 5.5034e-04 - val_accuracy: 0.9862 - val_f1_score: 0.4862 - val_loss: 0.0684
Epoch 13/20
456/499 ━━━━━━━━━━━━━━━━━━━━ 2:11 3s/step - accuracy: 0.9754 - f1_score: 0.6070 - loss: 5.3376e-04

In [ ]:
model.load_weights('/content/drive/MyDrive/seminar2/best_4_bilstm_toxic.h5')

In [ ]:
test_df=pd.read_csv('/content/drive/MyDrive/seminar2/dataset/test.csv')

In [ ]:
test_df['clean_text']=test_df['comment_text'].map(clean_text)

In [ ]:
test_seq = tokenizer.texts_to_sequences(test_df['clean_text'])
test_pad = pad_sequences(test_seq,maxlen=MAX_LEN, padding='post',truncating='post')

In [ ]:
predict=model.predict(test_pad)

499/499 ━━━━━━━━━━━━━━━━━━━━ 1148s 2s/step


In [ ]:
gold=test_df[LABELS].values

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(gold,np.round(predict))

0.9197894472991603

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(gold,np.round(predict),target_names=LABELS,digits=4))

               precision    recall  f1-score   support

        toxic     0.8385    0.7368    0.7844      1501
 severe_toxic     0.5283    0.1879    0.2772       149
      obscene     0.8236    0.7676    0.7946       809
       threat     0.0000    0.0000    0.0000        47
       insult     0.7260    0.6788    0.7016       769
identity_hate     0.0000    0.0000    0.0000       118

    micro avg     0.8004    0.6711    0.7300      3393
    macro avg     0.4861    0.3952    0.4263      3393
 weighted avg     0.7551    0.6711    0.7077      3393
  samples avg     0.0625    0.0591    0.0580      3393



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_